# First-Step-Contraction Replacement With MPC Upstream Controller And Bounded Frozen-dhat Target

This notebook keeps the same first-step-contraction MPC workflow and debug/export path as the maintained upstream-MPC notebook, but replaces the refined Step A steady-state target with the bounded frozen-dhat target developed from the offset-free steady-state analysis.

In [ ]:
import os
import numpy as np

from IPython.display import Image, display
from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic
from Simulation.mpc import MpcSolver, compute_observer_gain
from Simulation.run_mpc_first_step_contraction import run_mpc_first_step_contraction
from Simulation.system_functions import PolymerCSTR
from Lyapunov.safety_debug import build_safety_filter_run_bundle, save_safety_filter_debug_artifacts
from analysis.steady_state_debug_analysis import (
    analyze_offsetfree_rollout,
    save_offsetfree_ss_debug_artifacts,
)
from utils.scaling_helpers import apply_min_max
from utils.td3_helpers import load_and_prepare_system_data

In [ ]:
# Polymer CSTR setup
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])
delta_t = 0.5

steady_states = {"ss_inputs": system_steady_state_inputs.copy()}
cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)
steady_states["y_ss"] = cstr_ss.y_ss.copy()
# Match the original MPC notebook convention: the plant object stays in physical coordinates.
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)

u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])
setpoint_y_phys = np.array([[4.5, 324.0], [3.4, 321.0]])

system_dict_path = os.path.join("Data", "system_dict")
augmentation_style = "rawlings"
augmentation_mode = "mixed_B_I"

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y_phys,
    u_min=u_min,
    u_max=u_max,
    system_dict_path=system_dict_path,
    augmentation_style=augmentation_style,
    augmentation_mode=augmentation_mode,
)

A = system_data["A"]
B = system_data["B"]
C = system_data["C"]
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
Bd_used = system_data["Bd_used"]
Cd_used = system_data["Cd_used"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(setpoint_y_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"],
    data_min[inputs_number:],
    data_max[inputs_number:],
)

n_tests = 2
set_points_len = 400
TEST_CYCLE = [False, False]
warm_start = 0

poles = np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
                  0.42900673, 0.4228262, 0.96916776, 0.91230187])
L = compute_observer_gain(A_aug, C_aug, poles)

In [ ]:
# MPC configuration and reward
inputs_number = int(B_aug.shape[1])
predict_h = 9
cont_h = 3
u_ss = apply_min_max(steady_states["ss_inputs"], data_min[:inputs_number], data_max[:inputs_number])
b_min = apply_min_max(u_min, data_min[:inputs_number], data_max[:inputs_number])
b_max = apply_min_max(u_max, data_min[:inputs_number], data_max[:inputs_number])
b1 = (b_min[0] - u_ss[0], b_max[0] - u_ss[0])
b2 = (b_min[1] - u_ss[1], b_max[1] - u_ss[1])
bnds = (b1, b2) * cont_h
cons = []
IC_opt = np.zeros(inputs_number * cont_h)

MPC_obj = MpcSolver(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([5.0, 1.0]),
    R_in=np.array([1.0, 1.0]),
    NP=predict_h,
    NC=cont_h,
)

reward_config, reward_fn = make_reward_fn_mpc_quadratic(
    Q_diag=np.array([5.0, 1.0]),
    R_diag=np.array([1.0, 1.0]),
)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92

Qs_tgt_diag = 1e8 * np.asarray(MPC_obj.Q_out, float)
Ru_tgt_diag = 1.0 * np.ones(MPC_obj.B.shape[1])

In [ ]:
# Run MPC upstream + first-step-contraction replacement with bounded frozen-dhat target
u_ref_weight = 1.0

frozen_target_config = {
    "solver_mode": "auto",
    "cond_warn_threshold": 1e8,
    "residual_warn_threshold": 1e-8,
    "rank_tol": None,
    "box_bound_tol": 1e-9,
    "box_use_reduced_first": True,
    "u_ref_weight": u_ref_weight,
}

run_config = {
    "rho_lyap": 0.98,
    "lyap_eps": 1e-9,
    "lyap_tol": 1e-10,
    "target_generation_mode": "bounded_frozen_dhat",
    "frozen_target_config": frozen_target_config,
    "mpc_target_policy": "raw_setpoint",
    "tracking_target_policy": "raw_setpoint",
    "selector_H": None,
    "target_backup_policy": "last_valid",
    "selector_warm_start": True,
    "reuse_mpc_solution_as_ic": False,
    "reset_system_on_entry": True,
    "first_step_contraction_on": True,
}
print("Using bounded frozen-dhat steady-state target with first-step-contraction MPC replacement")
print("Bounded target regularization weight ||u_s - u_prev||^2:", u_ref_weight)

# Recreate the plant before each run so repeated executions match the baseline MPC notebook.
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)

results = run_mpc_first_step_contraction(
    system=cstr,
    MPC_obj=MPC_obj,
    y_sp_scenario=y_sp_scenario,
    n_tests=n_tests,
    set_points_len=set_points_len,
    steady_states=steady_states,
    IC_opt=IC_opt,
    bnds=bnds,
    cons=cons,
    warm_start=warm_start,
    L=L,
    data_min=data_min,
    data_max=data_max,
    test_cycle=TEST_CYCLE,
    reward_fn=reward_fn,
    nominal_qi=nominal_qi,
    nominal_qs=nominal_qs,
    nominal_ha=nominal_hA,
    qi_change=qi_change,
    qs_change=qs_change,
    ha_change=ha_change,
    mode="disturb",
    rho_lyap=run_config["rho_lyap"],
    lyap_eps=run_config["lyap_eps"],
    lyap_tol=run_config["lyap_tol"],
    Qy_track_diag=None,
    Rmove_diag=None,
    Qs_tgt_diag=Qs_tgt_diag,
    Ru_tgt_diag=Ru_tgt_diag,
    selector_H=run_config["selector_H"],
    mpc_target_policy=run_config["mpc_target_policy"],
    tracking_target_policy=run_config["tracking_target_policy"],
    target_backup_policy=run_config["target_backup_policy"],
    selector_warm_start=run_config["selector_warm_start"],
    reuse_mpc_solution_as_ic=run_config["reuse_mpc_solution_as_ic"],
    reset_system_on_entry=run_config["reset_system_on_entry"],
    first_step_contraction_on=run_config["first_step_contraction_on"],
    target_generation_mode=run_config["target_generation_mode"],
    frozen_target_config=run_config["frozen_target_config"],
)

bundle = build_safety_filter_run_bundle(
    source="mpc_first_step_replacement_bounded_frozen_dhat",
    results=results,
    steady_states=steady_states,
    config=run_config,
    min_max_dict=min_max_dict,
    data_min=data_min,
    data_max=data_max,
    extra={"reward_config": reward_config},
)

debug_dir = save_safety_filter_debug_artifacts(
    bundle=bundle,
    directory=os.path.join(os.getcwd(), "Data", "debug_exports"),
    prefix_name="mpc_first_step_replacement_bounded_frozen_dhat",
    save_plots=True,
)

bundle["summary"], debug_dir

In [ ]:
# Frozen-dhat steady-state analysis sidecar and inline plots
steady_state_rollout = {
    "y_mpc": bundle["y_system"],
    "u_mpc": bundle["u_applied_phys"],
    "xhatdhat": bundle["xhatdhat"],
    "y_sp": bundle["y_sp"],
    "nFE": bundle["nFE"],
    "delta_t": delta_t,
    "u_prev0_dev": np.zeros(MPC_obj.B.shape[1], dtype=float),
}
steady_state_analysis_config = {
    "enabled": True,
    "solver_mode": "auto",
    "cond_warn_threshold": 1.0e8,
    "residual_warn_threshold": 1.0e-8,
    "enable_box_analysis": True,
    "box_bound_tol": 1.0e-9,
    "box_use_reduced_first": True,
    "u_ref_weight": frozen_target_config["u_ref_weight"],
    "box_event_window_radius": 5,
    "box_dhat_event_threshold": 5.0e-2,
    "box_max_event_plots": 6,
    "tail_window_samples": 20,
    "save_csv": True,
    "save_plots": True,
    "sample_table_stride": 20,
    "case_name": "first_step_contraction_bounded_frozen_dhat",
}
steady_state_bundle = analyze_offsetfree_rollout(
    rollout=steady_state_rollout,
    system_data=system_data,
    steady_states=steady_states,
    analysis_config=steady_state_analysis_config,
)
steady_state_debug_dir = save_offsetfree_ss_debug_artifacts(
    steady_state_bundle,
    directory=os.path.join(os.getcwd(), "Data", "standard_lyap_first_step_contraction", "steady_state_debug"),
)
print("Saved steady-state analysis artifacts to:", steady_state_debug_dir)
print("Summary stats:", steady_state_bundle["summary_stats"])
if steady_state_bundle.get("box_analysis") is not None:
    print("Box summary:", steady_state_bundle["box_analysis"]["overall_summary"])

plot_names = [
    "outputs_vs_target.png",
    "inputs_vs_targets_dev.png",
    "reduced_rhs_vs_Gu.png",
    "tail_last_20_samples_overview.png",
]
for plot_name in plot_names:
    plot_path = os.path.join(steady_state_debug_dir, plot_name)
    if os.path.exists(plot_path):
        print(plot_name)
        display(Image(filename=plot_path))